In [ ]:
import ast
from nltk import sent_tokenize
import pandas as pd
from typing import List

from dap_job_quality import BUCKET_NAME
from dap_job_quality.getters.data_getters import load_s3_data, save_to_s3
from dap_job_quality.getters.ojo_getters import get_ojo_sample
from dap_job_quality.utils import text_cleaning as tc

In [ ]:
def find_search_terms(text: str, search_terms_list: List[str]) -> List[str]:
    """Find search terms in text.

    Args:
        text (str): The input text.
        search_terms_list (List[str]): List of search terms.

    Returns:
        List[str]: List of search terms found in the text.
    """
    found_terms = [term for term in search_terms_list if term in text]
    return found_terms

In [ ]:
categories = pd.read_csv("category_representation - category_representation.csv")
categories = categories.dropna(subset=["search_terms"])
categories["search_terms"] = categories["search_terms"].apply(ast.literal_eval)
categories = categories.explode("search_terms")

In [ ]:
search_terms_list = (
            categories["search_terms"].unique().tolist()
        )

In [ ]:
categories[['search_terms', 'subcategory']]

In [ ]:
train_ids = load_s3_data(BUCKET_NAME, 'job_quality/sentence_classifier/inputs/labelled/train_ids.parquet')['id']
val_ids = load_s3_data(BUCKET_NAME, 'job_quality/sentence_classifier/inputs/labelled/val_ids.parquet')['id']
test_ids = load_s3_data(BUCKET_NAME, 'job_quality/sentence_classifier/inputs/labelled/test_ids.parquet')['id']

In [ ]:
all_labelled_ids = list(train_ids) + list(val_ids) + list(test_ids)
len(all_labelled_ids)

In [ ]:
ojo_sample = get_ojo_sample()

In [ ]:
len(ojo_sample)

In [ ]:
ojo_sample.head()

In [ ]:
ojo_sample = ojo_sample[~ojo_sample['id'].isin(all_labelled_ids)]

In [ ]:
len(ojo_sample)

In [ ]:
ojo_sample['description_cleaned'] = ojo_sample['description'].apply(tc.clean_text).str.replace("[", "").str.replace("]", "").str.strip()

In [ ]:
ojo_sample[['description', 'description_cleaned']].head().to_csv("check_data.csv")

In [ ]:
ojo_sample['sentences'] = ojo_sample['description_cleaned'].apply(sent_tokenize)

In [ ]:
ojo_sample.head()

In [ ]:
ojo_sample_long = ojo_sample.explode("sentences")

In [ ]:
ojo_sample_long["search_terms"] = ojo_sample_long["sentences"].apply(
            lambda text: find_search_terms(text, search_terms_list)
        )

In [ ]:
ojo_sample_filtered = ojo_sample_long[
            ojo_sample_long["search_terms"].apply(bool)
        ].reset_index(drop=True)

In [ ]:
len(ojo_sample_filtered)

In [ ]:
ojo_sample_filtered_long = ojo_sample_filtered.explode("search_terms")
ojo_sample_filtered_long = pd.merge(ojo_sample_filtered_long, categories[['search_terms', 'subcategory']], on='search_terms', how='left')

In [ ]:
SENTS_PER_CAT = 50
sampled_ojo = ojo_sample_filtered_long.groupby('subcategory').apply(lambda x: x.sample(min(len(x), SENTS_PER_CAT), random_state=42)).reset_index(drop=True)

In [ ]:
sampled_ojo.head(50)

In [ ]:
sampled_ojo_deduplicated = sampled_ojo.drop_duplicates(subset=['id', 'sentences'])

In [ ]:
len(sampled_ojo) - len(sampled_ojo_deduplicated)

In [ ]:
len(sampled_ojo_deduplicated)

In [ ]:
sampled_ojo_deduplicated

In [ ]:
sampled_ojo_deduplicated['subcategory'].value_counts()

In [ ]:
save_to_s3(BUCKET_NAME, sampled_ojo_deduplicated, 'job_quality/sentence_classifier/inputs/labelling/additional_green_jobs_examples_20240704.csv')
# sampled_ojo_deduplicated.to_csv("additional_green_jobs_examples_20240704.csv", index=False)